# VoxCPM2 — TTS tiếng Việt với giọng của bạn

**Model**: VoxCPM2 (2B) — openbmb/VoxCPM2  
**Mục tiêu**: Sinh audio bằng giọng clone từ file ghi âm của bạn, tối ưu RTF  
**ASR**: Zipformer-transducer (sherpa-onnx)  
**Metrics**: WER, CER, UTMOS, SIM-o, RTF  

## Chuẩn bị
1. **Ghi âm giọng bạn**: 5–10 giây, đọc rõ ràng 1 câu tiếng Việt, ít nhiễu, WAV/MP3  
2. **Upload lên Kaggle**: Add Data → Upload → đặt tên dataset (vd: `my-voice`)  
3. File sẽ nằm tại `/kaggle/input/my-voice/ten_file.wav`  
4. Sửa `MY_VOICE_PATH` và `MY_VOICE_TEXT` ở cell bên dưới

In [ ]:
# === CÀI ĐẶT ===
!pip install -q jiwer soundfile librosa pandas matplotlib scipy sherpa-onnx transformers safetensors torchaudio
!pip install -q voxcpm

import os, json, time, glob, re, unicodedata, shutil
import numpy as np
from pathlib import Path

OUT_ROOT   = os.path.expanduser("~/voxcpm2_eval")
AUDIO_DIR  = f"{OUT_ROOT}/audio"
RESULT_DIR = f"{OUT_ROOT}/results"
TIMING_DIR = f"{OUT_ROOT}/timings"
ASR_DIR    = f"{OUT_ROOT}/asr_models"
OUTPUT_DIR = "/kaggle/working/output"

for d in (OUT_ROOT, AUDIO_DIR, RESULT_DIR, TIMING_DIR, ASR_DIR, OUTPUT_DIR):
    os.makedirs(d, exist_ok=True)

import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    torch.backends.cudnn.benchmark = True
print("Output:", OUTPUT_DIR)

## 1. Ref Audio — Giọng của bạn
Sửa 2 dòng dưới đây cho đúng đường dẫn và nội dung câu bạn đã đọc.

In [ ]:
# ============================================================
# SỬA 2 DÒNG NÀY
# ============================================================
MY_VOICE_PATH = "/kaggle/input/my-voice/giong_cua_toi.wav"  # <-- đường dẫn file ghi âm
MY_VOICE_TEXT = "Xin chào, đây là giọng nói của tôi."       # <-- nội dung câu đã đọc
# ============================================================

import soundfile as sf
import librosa

assert os.path.exists(MY_VOICE_PATH), f"Không tìm thấy file: {MY_VOICE_PATH}\nKiểm tra lại đường dẫn."

# Kiểm tra chất lượng ref audio
wave, sr = librosa.load(MY_VOICE_PATH, sr=None, mono=True)
dur = len(wave) / sr
rms = np.sqrt(np.mean(wave**2))

print(f"File: {MY_VOICE_PATH}")
print(f"Duration: {dur:.1f}s | SR: {sr} Hz | RMS: {rms:.4f}")
print(f"Text: {MY_VOICE_TEXT}")

if dur < 3:
    print("⚠ Quá ngắn (<3s). Chất lượng clone có thể kém. Nên ghi âm dài hơn.")
elif dur > 15:
    print("⚠ Quá dài (>15s). VoxCPM2 hoạt động tốt nhất với ref 5-10s.")
else:
    print("OK — độ dài phù hợp.")

if rms < 0.01:
    print("⚠ Âm lượng rất nhỏ. Kiểm tra lại file ghi âm.")

REF_AUDIO_PATH = MY_VOICE_PATH
REF_AUDIO_TEXT = MY_VOICE_TEXT

## 2. ASR (Zipformer)

In [ ]:
# === TẢI ASR MODELS ===
import sherpa_onnx, urllib.request, tarfile

BASE = "https://github.com/k2-fsa/sherpa-onnx/releases/download/asr-models"
PKGS = {
    "zipformer_vi_70k": "sherpa-onnx-zipformer-vi-2025-04-20",
    "zipformer_vi_30m": "sherpa-onnx-zipformer-vi-30M-int8-2026-02-09",
}

for key, name in PKGS.items():
    dst = f"{ASR_DIR}/{name}"
    if os.path.isdir(dst):
        print("đã có:", name); continue
    print("tải:", name, "...")
    arc = f"{ASR_DIR}/{name}.tar.bz2"
    urllib.request.urlretrieve(f"{BASE}/{name}.tar.bz2", arc)
    with tarfile.open(arc, 'r:bz2') as tf:
        tf.extractall(ASR_DIR)
    os.remove(arc)
    print("  xong")

In [ ]:
# === BUILD RECOGNIZER ===
import numpy as np, librosa, soundfile as sf

def _find(d, prefix):
    files = [f for f in os.listdir(d) if f.startswith(prefix) and f.endswith(".onnx")]
    if not files:
        raise FileNotFoundError(f"Không thấy {prefix}*.onnx trong {d}")
    int8 = sorted([f for f in files if "int8" in f])
    return os.path.join(d, int8[0] if int8 else sorted(files)[0])

def build_recognizer(pkg_key):
    name = PKGS[pkg_key]
    d = f"{ASR_DIR}/{name}"
    return sherpa_onnx.OfflineRecognizer.from_transducer(
        encoder=_find(d, "encoder"),
        decoder=_find(d, "decoder"),
        joiner=_find(d, "joiner"),
        tokens=os.path.join(d, "tokens.txt"),
        num_threads=2, sample_rate=16000, feature_dim=80,
        decoding_method="greedy_search",
    )

def transcribe(recognizer, wav_path):
    wave, _ = librosa.load(wav_path, sr=16000, mono=True)
    s = recognizer.create_stream()
    s.accept_waveform(16000, wave.astype(np.float32))
    recognizer.decode_stream(s)
    return s.result.text

ASR_MAIN  = build_recognizer("zipformer_vi_70k")
ASR_CROSS = build_recognizer("zipformer_vi_30m")
print("ASR loaded (Zipformer, KHÔNG Whisper)")

# Test ASR trên ref audio của bạn
hyp = transcribe(ASR_MAIN, REF_AUDIO_PATH)
print(f"ASR nghe thấy: {hyp}")
print(f"Bạn đọc:       {REF_AUDIO_TEXT}")

## 3. Text Normalizer + Scoring

In [ ]:
# === NUMBER TO WORDS + NORMALIZER ===
_ONES = ["không","một","hai","ba","bốn","năm","sáu","bảy","tám","chín"]

def _read_3(n, full):
    tr, ch, dv = n // 100, (n // 10) % 10, n % 10
    out = []
    if tr or full: out += [_ONES[tr], "trăm"]
    if ch == 0 and dv:
        if tr or full: out.append("linh")
        out.append(_ONES[dv])
    elif ch == 1:
        out.append("mười")
        if dv == 5: out.append("lăm")
        elif dv: out.append(_ONES[dv])
    elif ch > 1:
        out += [_ONES[ch], "mươi"]
        if dv == 1: out.append("mốt")
        elif dv == 4: out.append("tư")
        elif dv == 5: out.append("lăm")
        elif dv: out.append(_ONES[dv])
    return " ".join(out).strip()

_SCALES = ["", "nghìn", "triệu", "tỷ"]

def num_to_words(n):
    if n == 0: return "không"
    if n < 0:  return "âm " + num_to_words(-n)
    groups = []
    while n: groups.append(n % 1000); n //= 1000
    parts = []
    for i in range(len(groups) - 1, -1, -1):
        if groups[i] == 0: continue
        parts.append(_read_3(groups[i], full=(i != len(groups) - 1)))
        if _SCALES[i % 4]: parts.append(_SCALES[i % 4])
    return " ".join(parts)

def digits_to_words(s):
    return " ".join(_ONES[int(c)] for c in s if c.isdigit())

ABBREV = {
    "tp.hcm":"thành phố hồ chí minh", "tphcm":"thành phố hồ chí minh",
    "ubnd":"ủy ban nhân dân", "thpt":"trung học phổ thông",
    "tnhh":"trách nhiệm hữu hạn", "gs":"giáo sư", "ts":"tiến sĩ",
    "pgs":"phó giáo sư", "ths":"thạc sĩ", "cn":"cử nhân",
    "btc":"ban tổ chức", "hđqt":"hội đồng quản trị",
    "gdp":"tổng sản phẩm quốc nội",
}

_PUNCT = re.compile(r"[^\w\s]", flags=re.UNICODE)
_WS    = re.compile(r"\s+")
_NUM   = re.compile(r"\d[\d.,]*")

def normalize_v1(s, expand_numbers=True):
    s = unicodedata.normalize("NFC", str(s)).lower()
    s = s.replace("<unk>", " ").replace("\u2026", " ")
    for k in sorted(ABBREV, key=len, reverse=True):
        s = re.sub(rf"(?<!\w){re.escape(k)}(?!\w)", " " + ABBREV[k] + " ", s)
    s = s.replace("%", " ph\u1ea7n tr\u0103m ")
    if expand_numbers:
        def _rep(m):
            raw = m.group(0)
            if re.fullmatch(r"\d{1,3}(\.\d{3})+", raw):
                return " " + num_to_words(int(raw.replace(".", ""))) + " "
            if re.fullmatch(r"\d{1,3}(,\d{3})+", raw):
                return " " + num_to_words(int(raw.replace(",", ""))) + " "
            if re.fullmatch(r"\d+,\d+", raw):
                a, b = raw.split(",")
                return f" {num_to_words(int(a))} ph\u1ea9y {digits_to_words(b)} "
            if re.fullmatch(r"\d+\.\d+", raw):
                a, b = raw.split(".")
                return f" {num_to_words(int(a))} ch\u1ea5m {digits_to_words(b)} "
            digits = re.sub(r"\D", "", raw)
            if not digits: return " "
            if len(digits) > 6: return " " + digits_to_words(digits) + " "
            return " " + num_to_words(int(digits)) + " "
        s = _NUM.sub(_rep, s)
    s = _PUNCT.sub(" ", s)
    return _WS.sub(" ", s).strip()

import jiwer

def score_pair(ref_variants, hyp):
    h = normalize_v1(hyp)
    best = None
    for ref in ref_variants:
        r = normalize_v1(ref)
        if not r: continue
        w, c = jiwer.wer(r, h), jiwer.cer(r, h)
        if best is None or w < best[0]:
            best = (w, c, r, h)
    return best if best else (1.0, 1.0, "", h)

print("Normalizer + Scoring OK (no G2P, model handles English)")


## 4. Prompts (100+ câu, bao gồm mix EN/VN)

In [ ]:
# === PROMPTS ===
D = []
def add(pid, cat, text, spoken=None, alts=None, stress=False):
    D.append({"id": pid, "category": cat, "text": text,
              "spoken": spoken if spoken is not None else text,
              "alts": alts or [], "stress": stress})

add("everyday_01","everyday","Hôm nay thời tiết khá đẹp, rất thích hợp để đi dạo công viên.")
add("everyday_02","everyday","Cà phê sữa đá là thức uống yêu thích của rất nhiều người Việt Nam.")
add("everyday_03","everyday","Chiều nay mình phải ghé siêu thị mua ít đồ dùng cho gia đình.")
add("everyday_04","everyday","Con đường trước nhà vừa được sửa lại nên đi êm hơn hẳn.")
add("everyday_05","everyday","Bữa tối nay mẹ nấu canh chua cá lóc và thịt kho tiêu.")
add("everyday_06","everyday","Mình thường dậy lúc sáu giờ sáng để tập thể dục trước khi đi làm.")
add("everyday_07","everyday","Cuối tuần này cả nhà dự định về quê thăm ông bà nội.")
add("everyday_08","everyday","Quán phở đầu ngõ mở cửa từ sớm và lúc nào cũng đông khách.")
add("everyday_09","everyday","Trời trở lạnh rồi, nhớ mặc thêm áo khoác khi ra ngoài nhé.")
add("everyday_10","everyday","Tôi vừa đọc xong một cuốn sách rất hay về thói quen hằng ngày.")

add("question_01","question","Bạn đã ăn cơm chưa? Nếu chưa thì mình cùng đi ăn nhé.")
add("question_02","question","Tại sao hôm nay trời lại mưa to như vậy nhỉ?")
add("question_03","question","Cho mình hỏi đường ra bến xe miền Đông đi lối nào ạ?")
add("question_04","question","Anh có biết cuộc họp sáng mai bắt đầu lúc mấy giờ không?")
add("question_05","question","Em đã nộp báo cáo cho phòng nhân sự chưa vậy?")

add("emotion_01","emotion","Wow, đội tuyển vừa ghi bàn thắng đẹp quá!")
add("emotion_02","emotion","Trời ơi, xem pháo hoa tối qua đẹp không thể tin nổi!")
add("emotion_03","emotion","Chúc mừng bạn nhé, cuối cùng thì công sức cũng được đền đáp rồi!")
add("emotion_04","emotion","Đừng lo, mọi chuyện rồi sẽ ổn thôi, tôi luôn ở đây với bạn.")
add("emotion_05","emotion","Hạnh phúc đôi khi chỉ là được về nhà ăn cơm cùng gia đình.")

add("number_01","number","Hóa đơn hôm nay là 1.250.000 đồng, giảm 15% so với tuần trước.",
    "Hóa đơn hôm nay là một triệu hai trăm năm mươi nghìn đồng, giảm mười lăm phần trăm so với tuần trước.")
add("number_02","number","Nhiệt độ ngoài trời hiện tại là 32 độ C, độ ẩm 78%.",
    "Nhiệt độ ngoài trời hiện tại là ba mươi hai độ xê, độ ẩm bảy mươi tám phần trăm.")
add("number_03","number","Số điện thoại liên hệ của tôi là 0912 345 678.",
    "Số điện thoại liên hệ của tôi là không chín một hai ba bốn năm sáu bảy tám.")
add("number_04","number","Doanh thu quý ba đạt 47,5 tỷ đồng, tăng 8,2% so với cùng kỳ.",
    "Doanh thu quý ba đạt bốn mươi bảy phẩy năm tỷ đồng, tăng tám phẩy hai phần trăm so với cùng kỳ.")
add("number_05","number","Lớp tôi có 42 sinh viên, trong đó 18 bạn nữ và 24 bạn nam.",
    "Lớp tôi có bốn mươi hai sinh viên, trong đó mười tám bạn nữ và hai mươi bốn bạn nam.")

add("abbrev_01","abbrev","TP.HCM và Hà Nội là hai thành phố lớn nhất Việt Nam.",
    "Thành phố Hồ Chí Minh và Hà Nội là hai thành phố lớn nhất Việt Nam.")
add("abbrev_02","abbrev","UBND tỉnh vừa ban hành quyết định mới về quy hoạch đô thị.",
    "Ủy ban nhân dân tỉnh vừa ban hành quyết định mới về quy hoạch đô thị.")
add("abbrev_03","abbrev","Công ty TNHH Ánh Dương tuyển thêm 20 nhân viên kinh doanh.",
    "Công ty trách nhiệm hữu hạn Ánh Dương tuyển thêm hai mươi nhân viên kinh doanh.")

add("long_01","long","Dự án tốt nghiệp của tôi tập trung vào việc đánh giá và triển khai mô hình chuyển văn bản thành giọng nói cho tiếng Việt, nhằm phục vụ mục đích nghiên cứu và ứng dụng thực tế.")
add("long_02","long","Hệ thống sẽ tự động gửi thông báo tới địa chỉ đã đăng ký ngay khi đơn hàng được xác nhận, và người dùng có thể theo dõi toàn bộ hành trình vận chuyển ngay trên ứng dụng di động.")
add("long_03","long","Trước khi đưa mô hình vào môi trường thực tế, nhóm nghiên cứu đã tiến hành kiểm thử trên nhiều tập dữ liệu khác nhau, đo lường cẩn thận từng chỉ số và ghi nhận toàn bộ trường hợp lỗi.")

add("news_01","news","Bộ Giáo dục và Đào tạo vừa công bố phương án thi tốt nghiệp cho năm học tới.")
add("news_02","news","Giá xăng trong nước tiếp tục được điều chỉnh tăng từ chiều nay.")
add("news_03","news","Ngành du lịch đặt mục tiêu đón lượng khách quốc tế cao kỷ lục trong năm nay.")

add("narration_01","narration","Ngày xửa ngày xưa, ở một ngôi làng nhỏ ven sông, có một cậu bé tên là Tí.")
add("narration_02","narration","Buổi chiều hôm ấy, gió thổi qua cánh đồng lúa chín vàng rực cả một vùng.")
add("narration_03","narration","Tiếng chuông chùa vang lên từ phía xa, ngân dài trong buổi sớm tinh mơ.")

add("tech_01","tech","Nhóm mình dùng Python và Docker để triển khai hệ thống.",
    alts=["Nhóm mình dùng pai thon và đốc cơ để triển khai hệ thống."], stress=True)
add("tech_02","tech","Máy chủ bị quá tải do lượng truy cập tăng đột biến trong giờ cao điểm.")

# --- code_mix: câu tiếng Việt xen từ tiếng Anh, dùng để soi lỗi phát âm loanword ---
add("code_mix_01","code_mix","Tôi mới học xong khóa Python cơ bản trên Coursera.",
    alts=["Tôi mới học xong khóa pai thon cơ bản trên cơ rờ sợ ra."], stress=True)
add("code_mix_02","code_mix","Repo này dùng Docker và deploy lên GitHub Actions.",
    alts=["Ri pâu này dùng đốc cơ và đì plói lên gịt hắp ách sần."], stress=True)
add("code_mix_03","code_mix","Nhớ check email trước khi họp lúc 9 giờ sáng nhé.",
    alts=["Nhớ chéc i meo trước khi họp lúc chín giờ sáng nhé."], stress=True)
add("code_mix_04","code_mix","Wifi ở quán này chậm quá, chắc phải restart lại router.",
    alts=["Quai phai ở quán này chậm quá, chắc phải ri sờ tạc lại rao tơ."], stress=True)
add("code_mix_05","code_mix","Cho mình xin cái link Google Drive để tải file lên nhé.",
    alts=["Cho mình xin cái linh gu gồ đờ rai vơ để tải phai lên nhé."], stress=True)
add("code_mix_06","code_mix","Video này được share trên Facebook mấy triệu lượt xem.",
    alts=["Vi đi ô này được se trên phây búc mấy triệu lượt xem."], stress=True)
add("code_mix_07","code_mix","App bị crash liên tục sau khi update bản mới.",
    alts=["Áp bị cờ rát liên tục sau khi ắp đết bản mới."], stress=True)
add("code_mix_08","code_mix","Laptop của tôi hết pin nên phải sạc bằng sạc dự phòng.",
    alts=["Láp tóp của tôi hết pin nên phải sạc bằng sạc dự phòng."], stress=True)

add("misc_01","misc","Trân trọng cảm ơn quý khách đã tin tưởng và sử dụng dịch vụ của chúng tôi.")
add("misc_02","misc","Chúc quý vị và gia đình một năm mới an khang thịnh vượng.")

PROMPTS = {d["id"]: d for d in D}
json.dump(D, open(f"{OUT_ROOT}/prompts.json","w",encoding="utf-8"), ensure_ascii=False, indent=2)

from collections import Counter
print(len(D), "câu |", dict(Counter(d["category"] for d in D)))

## 5. Load VoxCPM2 — Tối ưu RTF

**Các bước giảm RTF:**
1. Tắt `torch.compile` (warm-up trên T4 chậm vì không hỗ trợ bf16 native)  
2. Tắt denoiser (`load_denoiser=False`)  
3. `cudnn.benchmark = True` (đã set ở trên)  
4. Không cache CUDA graph

In [ ]:
# === LOAD VOXCPM2 — TỐI ƯU RTF ===

# Tắt torch.compile: T4 không hỗ trợ bf16 native, compile chậm + ngốn RAM
import torch._dynamo
torch._dynamo.config.suppress_errors = True
torch._dynamo.reset()

from voxcpm import VoxCPM

_vox = VoxCPM.from_pretrained("openbmb/VoxCPM2", load_denoiser=False)
VOX_SR = _vox.tts_model.sample_rate
print(f"VoxCPM2 loaded. SR={VOX_SR}")
print(f"Ref audio: {REF_AUDIO_PATH}")

In [ ]:
# === GENERATE + RUN BATCH ===
os.makedirs(f"{AUDIO_DIR}/voxcpm2", exist_ok=True)

def preprocess_for_tts(text):
    """Expand viết tắt, normalize whitespace. Giữ nguyên tiếng Anh cho model tự đọc."""
    s = text
    for k in sorted(ABBREV, key=len, reverse=True):
        expanded = ABBREV[k]
        if len(expanded.split()) >= 3:
            expanded = expanded + ","
        s = re.sub(rf"(?<!\w){re.escape(k)}(?!\w)", " " + expanded + " ", s, flags=re.IGNORECASE)
    return re.sub(r'\s+', ' ', s).strip()

def generate_voxcpm2(text):
    tts_text = preprocess_for_tts(text)
    wav = _vox.generate(
        text=tts_text,
        reference_wav_path=REF_AUDIO_PATH,
    )
    return np.asarray(wav, dtype=np.float32), VOX_SR

# Test
print("=== Test preprocess_for_tts (no G2P) ===")
_test = [
    "TP.HCM và Hà Nội là hai thành phố lớn nhất Việt Nam.",
    "UBND tỉnh vừa ban hành quyết định mới về quy hoạch đô thị.",
    "Số điện thoại liên hệ của tôi là 0912 345 678.",
    "Nhóm mình dùng Python và Docker để triển khai hệ thống.",
    "API này cần token từ OpenAI.",
    "App bị crash liên tục sau khi update bản mới.",
]
for s in _test:
    r = preprocess_for_tts(s)
    changed = " ← SỬA" if r != s else ""
    print(f"  {s}")
    print(f"  → {r}{changed}\n")

t0 = time.time()
_t, _s = generate_voxcpm2("TP.HCM có 1.250.000 người.")
dt = time.time() - t0
audio_dur = len(_t) / _s
print(f"Test OK: {audio_dur:.1f}s audio, {dt:.1f}s gen, RTF={dt/audio_dur:.2f}")
sf.write(f"{OUTPUT_DIR}/test_sample.wav", _t, _s)
del _t

# Run batch
timings = {}
tpath = f"{TIMING_DIR}/voxcpm2.json"
if os.path.exists(tpath):
    timings = json.load(open(tpath))

done = fail = 0
for pid, p in PROMPTS.items():
    out = f"{AUDIO_DIR}/voxcpm2/{pid}.wav"
    if os.path.exists(out) and pid in timings:
        continue
    try:
        t0 = time.time()
        audio, sr = generate_voxcpm2(p["text"])
        dt = time.time() - t0
        audio = np.asarray(audio, dtype=np.float32).squeeze()
        if audio.ndim > 1:
            audio = audio.mean(axis=0 if audio.shape[0] < audio.shape[1] else 1)
        sf.write(out, audio, sr)
        timings[pid] = {"gen_s": round(dt, 3),
                        "audio_s": round(len(audio) / sr, 3),
                        "rtf": round(dt / max(len(audio) / sr, 1e-6), 4)}
        done += 1
    except Exception as e:
        fail += 1
        print(f"lỗi {pid}: {type(e).__name__}: {e}")
    if (done + fail) % 10 == 0:
        json.dump(timings, open(tpath, "w"), indent=2)
        print(f"  ...{done+fail}/{len(PROMPTS)}")

json.dump(timings, open(tpath, "w"), indent=2)
rtfs = [v["rtf"] for v in timings.values()]
print(f"\nXong. Sinh {done}, lỗi {fail}, tổng {len(timings)}")
print(f"RTF trung bình: {np.mean(rtfs):.3f}")
print(f"RTF min/max: {np.min(rtfs):.3f} / {np.max(rtfs):.3f}")


In [ ]:
# === TEST VĂN BẢN DÀI ===
long_text = """Kính thưa quý vị và các bạn, hôm nay tôi xin trình bày về dự án tốt nghiệp 
của mình với chủ đề đánh giá và triển khai mô hình chuyển văn bản thành giọng nói cho 
tiếng Việt. Trong bối cảnh công nghệ trí tuệ nhân tạo đang phát triển mạnh mẽ, việc 
tổng hợp giọng nói tự nhiên từ văn bản đã trở thành một lĩnh vực nghiên cứu quan trọng, 
có ứng dụng rộng rãi trong nhiều ngành như giáo dục, y tế, giải trí và hỗ trợ người khuyết tật. 
Mô hình mà nhóm chúng tôi lựa chọn là VoxCPM2, một mô hình ngôn ngữ lớn với hai tỷ tham số, 
có khả năng tạo giọng nói tự nhiên bằng cách clone giọng từ một đoạn ghi âm tham chiếu ngắn 
chỉ từ năm đến mười giây. Quá trình đánh giá được thực hiện trên tập dữ liệu gồm hơn 
một trăm câu tiếng Việt thuộc nhiều thể loại khác nhau, bao gồm câu hội thoại thường ngày, 
câu hỏi, câu cảm thán, câu chứa số liệu, câu viết tắt, và cả câu dài phức tạp. 
Các chỉ số đánh giá bao gồm tỷ lệ lỗi từ, tỷ lệ lỗi ký tự, điểm chất lượng giọng nói UTMOS, 
độ tương đồng giọng nói SIM-o, và hệ số thời gian thực RTF. Kết quả cho thấy mô hình VoxCPM2 
đạt được chất lượng giọng nói khá tốt trên tiếng Việt, tuy nhiên vẫn còn một số hạn chế 
khi xử lý các từ mượn tiếng Anh và các câu có cấu trúc phức tạp. Xin cảm ơn quý vị đã lắng nghe."""

long_text = " ".join(long_text.split())
print(f"Độ dài: {len(long_text)} ký tự, ~{len(long_text.split())} từ\n")

# Sinh audio
t0 = time.time()
audio_long, sr_long = generate_voxcpm2(long_text)
dt = time.time() - t0
audio_dur = len(audio_long) / sr_long

print(f"Gen time:   {dt:.1f}s")
print(f"Audio time: {audio_dur:.1f}s")
print(f"RTF:        {dt/audio_dur:.4f}")
print(f"{'Nhanh hơn' if dt < audio_dur else 'Chậm hơn'} thời gian thực")

sf.write(f"{OUTPUT_DIR}/test_long_text.wav", audio_long, sr_long)
print(f"\nĐã lưu: {OUTPUT_DIR}/test_long_text.wav")

# ASR kiểm tra
hyp = transcribe(ASR_MAIN, f"{OUTPUT_DIR}/test_long_text.wav")
print(f"\nASR nghe được ({len(hyp.split())} từ):\n{hyp[:300]}...")

## 6. Đánh giá

In [ ]:
# === EVAL WER ===
import pandas as pd

rows = []
wavs = sorted(glob.glob(f"{AUDIO_DIR}/voxcpm2/*.wav"))
print(f"Chấm WER: {len(wavs)} file...")

for w in wavs:
    pid = Path(w).stem
    if pid not in PROMPTS: continue
    p = PROMPTS[pid]
    rv = [p["spoken"]] + list(p.get("alts", []))
    h1 = transcribe(ASR_MAIN, w)
    h2 = transcribe(ASR_CROSS, w)
    w1, c1, ref_used, hyp1 = score_pair(rv, h1)
    w2, c2, _, _ = score_pair(rv, h2)
    rows.append({"prompt_id": pid, "category": p["category"],
                 "stress": p["stress"], "ref": ref_used,
                 "hyp_main": hyp1, "wer": w1, "cer": c1,
                 "wer_cross": w2, "wer_agree": min(w1, w2)})

df_wer = pd.DataFrame(rows)
df_wer.to_csv(f"{RESULT_DIR}/wer_detail.csv", index=False)

core = df_wer[~df_wer.stress]
print(f"\n=== VoxCPM2 — WER (loại câu stress) ===")
print(f"WER:  {core.wer.mean():.4f}")
print(f"CER:  {core.cer.mean():.4f}")
print(f"WER (agree): {core.wer_agree.mean():.4f}")

print(f"\n=== WER theo nhóm câu ===")
cat_wer = core.groupby("category")["wer"].mean().round(4).sort_values()
print(cat_wer.to_string())

code_mix = df_wer[df_wer.category == "code_mix"]
if len(code_mix):
    print(f"\n=== code_mix (câu EN/VN, dùng phiên âm loanword) ===")
    print(code_mix[["prompt_id", "wer", "cer"]].to_string(index=False))
    print(f"WER trung bình code_mix: {code_mix.wer.mean():.4f}")

In [ ]:
# === UTMOS ===
predictor = torch.hub.load("tarepan/SpeechMOS:v1.2.0", "utmos22_strong", trust_repo=True)
predictor = predictor.to(DEVICE).eval()

utmos_rows = []
for w in sorted(glob.glob(f"{AUDIO_DIR}/voxcpm2/*.wav")):
    wave, _ = librosa.load(w, sr=16000, mono=True)
    with torch.no_grad():
        s = predictor(torch.from_numpy(wave).unsqueeze(0).to(DEVICE), 16000).item()
    utmos_rows.append({"prompt_id": Path(w).stem, "utmos": s})

df_utmos = pd.DataFrame(utmos_rows)
df_utmos.to_csv(f"{RESULT_DIR}/utmos_detail.csv", index=False)
print(f"UTMOS trung bình: {df_utmos.utmos.mean():.3f} ± {df_utmos.utmos.std():.3f}")

In [ ]:
# === SIM-o (Speaker Similarity với giọng của bạn) ===
from transformers import AutoFeatureExtractor, WavLMForXVector
import torch.nn.functional as F

_fe = AutoFeatureExtractor.from_pretrained("microsoft/wavlm-base-plus-sv")
_sv = WavLMForXVector.from_pretrained("microsoft/wavlm-base-plus-sv").to(DEVICE).eval()

def embed(wav_path):
    wave, _ = librosa.load(wav_path, sr=16000, mono=True)
    inp = _fe([wave], sampling_rate=16000, return_tensors="pt", padding=True)
    inp = {k: v.to(DEVICE) for k, v in inp.items()}
    with torch.no_grad():
        e = _sv(**inp).embeddings
    return F.normalize(e, dim=-1)

ref_emb = embed(REF_AUDIO_PATH)
sim_rows = []
for w in sorted(glob.glob(f"{AUDIO_DIR}/voxcpm2/*.wav")):
    sim = float(torch.cosine_similarity(ref_emb, embed(w)).item())
    sim_rows.append({"prompt_id": Path(w).stem, "sim_o": sim})

df_sim = pd.DataFrame(sim_rows)
df_sim.to_csv(f"{RESULT_DIR}/simo_detail.csv", index=False)
print(f"SIM-o trung bình: {df_sim.sim_o.mean():.4f} ± {df_sim.sim_o.std():.4f}")
print("(Cao hơn = giống giọng bạn hơn)")

In [ ]:
# === RTF ===
t = json.load(open(f"{TIMING_DIR}/voxcpm2.json"))
gen_total = sum(v["gen_s"] for v in t.values())
aud_total = sum(v["audio_s"] for v in t.values())
rtfs = [v["rtf"] for v in t.values()]

print(f"RTF gộp:          {gen_total/aud_total:.4f}")
print(f"RTF trung bình:   {np.mean(rtfs):.4f}")
print(f"RTF median:       {np.median(rtfs):.4f}")
print(f"RTF min / max:    {np.min(rtfs):.4f} / {np.max(rtfs):.4f}")
print(f"Tổng gen time:    {gen_total:.1f}s")
print(f"Tổng audio time:  {aud_total:.1f}s")
print(f"\nRTF < 1 = nhanh hơn thời gian thực ({DEVICE})")

## 7. Phân tích lỗi

In [ ]:
# === PHÂN TÍCH LỖI TỪNG CÂU ===
error_rows = []
for _, row in df_wer.iterrows():
    if row.wer == 0: continue
    ref_words = row.ref.split()
    hyp_words = row.hyp_main.split()
    try:
        out = jiwer.process_words([row.ref], [row.hyp_main])
    except: continue
    missing, extra, wrong = [], [], []
    for chunk in out.alignments[0]:
        if chunk.type == 'delete':
            for k in range(chunk.ref_start_idx, chunk.ref_end_idx):
                missing.append(ref_words[k])
        elif chunk.type == 'insert':
            for k in range(chunk.hyp_start_idx, chunk.hyp_end_idx):
                extra.append(hyp_words[k])
        elif chunk.type == 'substitute':
            for k in range(chunk.ref_start_idx, chunk.ref_end_idx):
                ref_w = ref_words[k] if k < len(ref_words) else '?'
                hyp_idx = chunk.hyp_start_idx + (k - chunk.ref_start_idx)
                hyp_w = hyp_words[hyp_idx] if hyp_idx < len(hyp_words) else '?'
                wrong.append(f"{ref_w}→{hyp_w}")
    detail = []
    if missing: detail.append(f"thiếu: {' '.join(missing)}")
    if extra: detail.append(f"thừa: {' '.join(extra)}")
    if wrong: detail.append(f"sai: {' '.join(wrong)}")
    error_rows.append({'prompt_id': row.prompt_id, 'category': row.category,
        'wer': row.wer, 'detail': ' | '.join(detail)})

df_errors = pd.DataFrame(error_rows)
df_errors.to_csv(f"{RESULT_DIR}/error_analysis.csv", index=False)

print(f"Số câu có lỗi: {len(df_errors)} / {len(df_wer)}")
print(f"\n=== TOP 10 CÂU TỆ NHẤT ===")
for _, r in df_errors.nlargest(10, 'wer').iterrows():
    print(f"  [{r.prompt_id}] WER={r.wer:.2f} | {r.detail[:120]}")

## 8. Báo cáo tổng hợp + Output files

In [ ]:
# === BÁO CÁO + COPY OUTPUT ===
print("="*68)
print("BÁO CÁO: VOXCPM2 TTS TIẾNG VIỆT VỚI GIỌNG CỦA BẠN")
print("="*68)
print(f"Model        : VoxCPM2 (2B) — openbmb/VoxCPM2")
print(f"Ref audio    : {REF_AUDIO_PATH}")
print(f"Ref text     : {REF_AUDIO_TEXT}")
print(f"Dataset      : {len(PROMPTS)} câu")
print(f"Loanword TTS : {'BẬT' if False else 'TẮT'} (phiên âm từ mượn EN)")
print(f"ASR          : Zipformer (KHÔNG Whisper)")
print(f"Device       : {DEVICE}")
if DEVICE == 'cuda':
    print(f"GPU          : {torch.cuda.get_device_name(0)}")

core = df_wer[~df_wer.stress]
print(f"\n--- Kết quả ---")
print(f"WER:         {core.wer.mean():.4f}")
print(f"CER:         {core.cer.mean():.4f}")
print(f"UTMOS:       {df_utmos.utmos.mean():.3f}")
print(f"SIM-o:       {df_sim.sim_o.mean():.4f}")
print(f"RTF:         {np.mean(rtfs):.4f}")
print(f"RTF < 1 = nhanh hơn thời gian thực")

# Copy tất cả output vào /kaggle/working/output
# Audio files
audio_out = f"{OUTPUT_DIR}/audio"
os.makedirs(audio_out, exist_ok=True)
for w in glob.glob(f"{AUDIO_DIR}/voxcpm2/*.wav"):
    shutil.copy2(w, audio_out)

# CSV files
for csv in glob.glob(f"{RESULT_DIR}/*.csv"):
    shutil.copy2(csv, OUTPUT_DIR)

# Timings
shutil.copy2(f"{TIMING_DIR}/voxcpm2.json", f"{OUTPUT_DIR}/timings.json")

# Prompts
shutil.copy2(f"{OUT_ROOT}/prompts.json", f"{OUTPUT_DIR}/prompts.json")

# Ref audio
shutil.copy2(REF_AUDIO_PATH, f"{OUTPUT_DIR}/ref_audio.wav")

# Summary JSON
summary = {
    "model": "VoxCPM2",
    "ref_audio": os.path.basename(REF_AUDIO_PATH),
    "ref_text": REF_AUDIO_TEXT,
    "n_prompts": len(PROMPTS),
    "loanword_phonetic": False  # model handles English natively,
    "wer": round(core.wer.mean(), 4),
    "cer": round(core.cer.mean(), 4),
    "utmos": round(df_utmos.utmos.mean(), 3),
    "sim_o": round(df_sim.sim_o.mean(), 4),
    "rtf_mean": round(np.mean(rtfs), 4),
    "rtf_median": round(np.median(rtfs), 4),
    "device": DEVICE,
    "gpu": torch.cuda.get_device_name(0) if DEVICE == 'cuda' else "N/A",
}
json.dump(summary, open(f"{OUTPUT_DIR}/summary.json", "w"), indent=2, ensure_ascii=False)

# Zip toàn bộ
shutil.make_archive(f"/kaggle/working/voxcpm2_output", "zip", OUTPUT_DIR)

print(f"\n--- Output files ---")
for f in sorted(os.listdir(OUTPUT_DIR)):
    p = os.path.join(OUTPUT_DIR, f)
    if os.path.isdir(p):
        n = len(os.listdir(p))
        print(f"  {f}/ ({n} files)")
    else:
        sz = os.path.getsize(p)
        print(f"  {f} ({sz/1024:.1f} KB)")

print(f"\nZip: /kaggle/working/voxcpm2_output.zip")
print("Tải về từ Output tab của Kaggle notebook.")